For building, training, and using a recommended model, use `Surprise`.

In [1]:
!pip install scikit-surprise

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.4/154.4 kB 3.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for scikit-surprise: filename=scikit_surprise-1.1.4-cp312-cp312-linux_x86_64.whl size=2554975 sha256=2d1b895b952a8169937dc58b3521c3cb42ecf1f231755d5ea2a95c89e0d34706
  Stored in directory: /root/.cache/pip/wheels/75/fa/bc/739bc2cb1fbaab6061854e6cfbb81a0ae52c92a502a7fa454b
Successfully built scikit-surprise


In [1]:
!pip install "numpy<2"

In [2]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split, cross_validate
from surprise.model_selection import train_test_split, cross_validate, GridSearchCV

from surprise import accuracy

Read `ctgan_gym.xls` file.

In [4]:
df = pd.read_csv('ctgan_gym.xls')

print(df.tail())

       exercise_name  sets  min_reps  max_reps  difficulty  age  gender  \
21453             28     3         4         1           1   33       0   
21454            103     3         1         4           1   36       1   
21455             21     3         8         0           1   20       1   
21456             62     4         8         2           1   24       1   
21457             73     3         2         2           1   16       1   

       experience_numeric  days_per_week  location  program_name  \
21453                   1              4         0            16   
21454                   1              5         0             2   
21455                   1              6         0             7   
21456                   2              6         1            11   
21457                   1              6         0             2   

       program_difficulty  program_days_per_week  program_goal  \
21453                   0                      5             3   
21454   

In [5]:
print(df.isnull().sum())

exercise_name            0
sets                     0
min_reps                 0
max_reps                 0
difficulty               0
age                      0
gender                   0
experience_numeric       0
days_per_week            0
location                 0
program_name             0
program_difficulty       0
program_days_per_week    0
program_goal             0
start_weight_kg          0
end_weight_kg            0
goal_achieved            0
user_rating              0
dtype: int64


Prepare the dataset for `SVD` model.

In [6]:
df['user_id'] = df.groupby(['age', 'gender', 'experience_numeric', 'days_per_week', 'program_goal','location']).ngroup()

In [7]:
df['combined_score'] = (
    df['user_rating'] * 0.6 +
    (df['goal_achieved'] == 2).astype(int) * 3 +
    (df['goal_achieved'] == 1).astype(int) * 1 +
    (df['end_weight_kg'] > df['start_weight_kg']).astype(int) * 0.5
)

min_score = df['combined_score'].min()
max_score = df['combined_score'].max()
df['rating'] = 1 + 4 * (df['combined_score'] - min_score) / (max_score - min_score)
df['rating'] = df['rating'].round(1)

Use `Reader` for setting the rating limits from 1 to 5.  
`Dataset.load_from_df` is used to convert a DataFrame into the format required by `Surprise`.


In [17]:
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(df[['user_id', 'program_name', 'user_rating']], reader)
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

`SVD` is popular for building recommendation models

In [18]:
model = SVD(n_factors=50, n_epochs=20, lr_all=0.005, reg_all=0.02)

model.fit(trainset)

In [19]:
predictions = model.test(testset)

rmse = accuracy.rmse(predictions)
mae = accuracy.mae(predictions)

RMSE: 0.4831
MAE:  0.4575


Search for the best hyperparameters for the model


In [20]:
param_grid = {
    'n_factors': [20, 50, 100],
    'n_epochs': [10, 20, 30],
    'lr_all': [0.002, 0.005, 0.01],
    'reg_all': [0.02, 0.05, 0.1]
}

gs = GridSearchCV(SVD, param_grid, measures=['rmse', 'mae'], cv=3)
gs.fit(data)

print("The best features:", gs.best_params['rmse'])
print("best RMSE:", gs.best_score['rmse'])

best_model = gs.best_estimator['rmse']

The best features: {'n_factors': 50, 'n_epochs': 20, 'lr_all': 0.002, 'reg_all': 0.02}
best RMSE: 0.4822271341598221


Evaluate the best model using cross-validation


In [21]:
cv_results = cross_validate(best_model, data, measures=['RMSE', 'MAE'], cv=5, verbose=True)

print(f"avg RMSE: {cv_results['test_rmse'].mean():.4f}")
print(f"Standard deviation RMSE: {cv_results['test_rmse'].std():.4f}")
print(f"avg MAE: {cv_results['test_mae'].mean():.4f}")

Evaluating RMSE, MAE of algorithm SVD on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.4802  0.4830  0.4820  0.4831  0.4817  0.4820  0.0010  
MAE (testset)     0.4622  0.4658  0.4638  0.4649  0.4634  0.4640  0.0012  
Fit time          0.30    0.31    0.17    0.18    0.21    0.23    0.06    
Test time         0.04    0.12    0.02    0.03    0.03    0.05    0.04    
avg RMSE: 0.4820
Standard deviation RMSE: 0.0010
avg MAE: 0.4640


Save the model

In [22]:
import pickle

with open('model.pkl', 'wb') as f:
    pickle.dump(best_model, f)